<table>
  <tr>
    <td style="text-align: center">
      <a href="https://colab.research.google.com/github/arvind-dhariwal/aeon-credit-gcp-workshop/blob/main/track1_platform_governance/02_Data_Engineering_Agent_Medallion_Transformation.ipynb">
        <img width="32px" src="https://www.gstatic.com/pantheon/images/bigquery/welcome_page/colab-logo.svg" alt="Google Colaboratory logo"><br> Open in Colab
      </a>
    </td>
    <td style="text-align: center">
      <a href="https://console.cloud.google.com/vertex-ai/colab/import/https:%2F%2Fraw.githubusercontent.com%2Farvind-dhariwal%2Faeon-credit-gcp-workshop%2Fmain%2Ftrack1_platform_governance%2F02_Data_Engineering_Agent_Medallion_Transformation.ipynb">
        <img width="32px" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" alt="Google Cloud Colab Enterprise logo"><br> Open in Colab Enterprise
      </a>
    </td>
    <td style="text-align: center">
      <a href="https://console.cloud.google.com/vertex-ai/workbench/deploy-notebook?download_url=https://raw.githubusercontent.com/arvind-dhariwal/aeon-credit-gcp-workshop/main/track1_platform_governance/02_Data_Engineering_Agent_Medallion_Transformation.ipynb">
        <img width="32px" src="https://storage.googleapis.com/github-repo/workbench-icon.svg" alt="Workbench logo"><br> Open in Workbench
      </a>
    </td>
    <td style="text-align: center">
      <a href="https://console.cloud.google.com/bigquery/import?url=https://github.com/arvind-dhariwal/aeon-credit-gcp-workshop/blob/main/track1_platform_governance/02_Data_Engineering_Agent_Medallion_Transformation.ipynb">
        <img src="https://www.gstatic.com/images/branding/gcpiconscolors/bigquery/v1/32px.svg" alt="BigQuery Studio logo"><br> Open in BigQuery Studio
      </a>
    </td>
    <td style="text-align: center">
      <a href="https://github.com/arvind-dhariwal/aeon-credit-gcp-workshop/blob/main/track1_platform_governance/02_Data_Engineering_Agent_Medallion_Transformation.ipynb">
        <img width="32px" src="https://raw.githubusercontent.com/primer/octicons/refs/heads/main/icons/mark-github-24.svg" alt="GitHub logo"><br> View on GitHub
      </a>
    </td>
  </tr>
</table>
<br clear="all"/>

---

# Track 1 (Notebook 02): Agentic Medallion Transformation (`acsm_bronze` → `acsm_silver` → `acsm_gold`)
**AEON Credit Service Malaysia (ACSM) — Google Agentic Data Cloud Workshop (`asia-southeast1` Singapore)**

---

## 🗺️ Notebook 02 Architecture & Medallion DAG (`acsm_bronze` → `acsm_silver` → `acsm_gold`)

![Track 1 Notebook 02 — Agentic Medallion Transformation Flow](https://raw.githubusercontent.com/arvind-dhariwal/aeon-credit-gcp-workshop/main/track1_platform_governance/images/notebook2_medallion_architecture_flow.png)

---
### 📋 Notebook 02 Step-by-Step Execution Summary
1. **Step 1**: Configure parameterized `PROJECT_ID` (auto-detects active project if blank) & `LOCATION = "asia-southeast1"` (Singapore) and enable the Data Engineering Agent APIs (`dataform.googleapis.com`, `cloudaicompanion.googleapis.com`).
2. **Step 0 (`%%bigquery` SQL — Pre-Flight Metadata Check)**: Confirm that **all 8 `acsm_bronze` source tables & views** (`6` Native Fact tables + `m3CIF` Lakehouse Iceberg view + `dimProduct` AWS Glue Federated Iceberg view) have **100% table and column descriptions** from `Mock Metadata.xlsx`.
3. **Step 2 (`%%bigquery` SQL)**: Verify row counts across all **8 `acsm_bronze` source tables & views** (`1,398,284` rows).
4. **Step 3 (`%%bigquery` SQL)**: Create the two target Medallion datasets (`acsm_silver` & `acsm_gold`) and pre-train the BigQuery ML model (`acsm_silver.model_delinquency_propensity`) for batch pipeline scoring.
5. **Step 4 (UI Guide + Prompt)**: Copy-paste the **Medallion Architecture + Incremental Load + Dataform Assertions + Batch BQML Prediction Prompt** into the **BigQuery Data Engineering Agent (`+ Create` → `Pipeline`)**.
6. **Step 5 (`%%bigquery` SQL)**: Transparent SQL execution of the **4 Silver tables + 2 Gold tables (`gold_aeon360_customer_profile` & `gold_aeon360_batch_ml_predictions`)** directly inside this notebook.
7. **Step 6 (`%%bigquery` SQL)**: Verify `acsm_silver` & `acsm_gold` row counts, run the **Dual-Run Financial Control Total Reconciliation Audit (`0.00 MYR Variance`)**, and preview the batch ML predictions.

---
## Step 1: Configure Project & Region (`asia-southeast1`) and Enable Data Engineering Agent APIs
Run the cell below to set your `PROJECT_ID` and `LOCATION` (`asia-southeast1` Singapore), load the `%bigquery` SQL magic extension, and enable the required APIs (`dataform.googleapis.com` and `cloudaicompanion.googleapis.com`) for the **BigQuery Data Engineering Agent**.

In [ ]:
# @title 1. Set Parameterized `PROJECT_ID` (Auto-Detects Active Project if Blank), Singapore Region (`asia-southeast1`), and Enable APIs
import os
import subprocess

PROJECT_ID = ""  # @param {type:"string"}
if not PROJECT_ID or PROJECT_ID.startswith("<"):
    PROJECT_ID = (
        os.environ.get("GOOGLE_CLOUD_PROJECT")
        or subprocess.check_output(["gcloud", "config", "get-value", "project"], text=True).strip()
    )
LOCATION = "asia-southeast1"  # @param {type:"string"}

os.environ["PROJECT_ID"] = PROJECT_ID
os.environ["LOCATION"] = LOCATION

# Load the native BigQuery SQL magic so all subsequent cells run pure SQL against $PROJECT_ID
%load_ext google.cloud.bigquery

# Enable BigQuery Pipelines (Dataform) and Gemini for Google Cloud (Data Engineering Agent)
!gcloud config set project $PROJECT_ID
!gcloud services enable bigquery.googleapis.com dataform.googleapis.com cloudaicompanion.googleapis.com --project=$PROJECT_ID
print(f"Ready! Active Project={PROJECT_ID} | Region={LOCATION} (Singapore)")


---
## Step 0 (Pre-Flight Metadata Audit): Confirm 100% Table & Column Descriptions Across All 8 `acsm_bronze` Tables & Views
Before invoking the **BigQuery Data Engineering Agent** to build the Medallion architecture, we confirm that **all 8 source tables and views in `acsm_bronze`** (`6` BigQuery Native Fact tables + the `m3CIF` GCP Lakehouse Iceberg view + the `dimProduct` AWS Glue Federated Iceberg view) have **100% Table-Level and Column-Level Descriptions** from `Mock Metadata.xlsx`.

> **Why this matters for the Data Engineering Agent**: Gemini in BigQuery reads `INFORMATION_SCHEMA` table and column descriptions (`OPTIONS(description=...)`) to understand business meaning, primary keys, date formats, and join keys from natural-language prompts!
> *(Note: This cell also automatically syncs all `Mock Metadata.xlsx` column descriptions onto the `acsm_bronze.m3CIF` and `acsm_bronze.dimProduct` views in case they were created as plain `SELECT *` views earlier.)*

In [ ]:
%%bigquery --project $PROJECT_ID --location $LOCATION
-- =============================================================================
-- Step 0 (Pure BigQuery SQL): Ensure & Audit 100% Table and Column Descriptions
-- Across All 8 `acsm_bronze` Source Tables & Views (`Mock Metadata.xlsx`)
-- =============================================================================

-- 1. Ensure `acsm_bronze.m3CIF` view has all 34 column descriptions from Mock Metadata.xlsx (Sheet: T7 - m3CIF)
BEGIN
  DECLARE m3_col_defs STRING;
  SET m3_col_defs = (
    WITH dict AS (
      SELECT 'Rcd_DT' AS col, 'Record refresh date [Source Data Type: date]' AS d UNION ALL
      SELECT 'CIF_ID', 'Customer ID [Source Data Type: varchar]' UNION ALL
      SELECT 'CIF_NM', 'Customer name [Source Data Type: varchar]' UNION ALL
      SELECT 'CIF_NM1', 'Customer name [Source Data Type: varchar]' UNION ALL
      SELECT 'CIF_NM2', 'Customer name [Source Data Type: varchar]' UNION ALL
      SELECT 'MaritalSts', 'MaritalStatus [Source Data Type: varchar]' UNION ALL
      SELECT 'Gender', 'Gender [Source Data Type: varchar]' UNION ALL
      SELECT 'Citizen', 'Citizen [Source Data Type: varchar]' UNION ALL
      SELECT 'State', 'State [Source Data Type: varchar]' UNION ALL
      SELECT 'Region', 'Region [Source Data Type: varchar]' UNION ALL
      SELECT 'Race', 'Race [Source Data Type: varchar]' UNION ALL
      SELECT 'NOB', 'Nature of business [Source Data Type: varchar]' UNION ALL
      SELECT 'HomeOwn', 'Customer homeowner category [Source Data Type: varchar]' UNION ALL
      SELECT 'HomePost', 'Customer home postcode [Source Data Type: varchar]' UNION ALL
      SELECT '_HomeAddr1', '_HomeAddr1 [Source Data Type: varchar]' UNION ALL
      SELECT '_HomeAddr2', '_HomeAddr2 [Source Data Type: varchar]' UNION ALL
      SELECT '_HomeAddr3', '_HomeAddr3 [Source Data Type: varchar]' UNION ALL
      SELECT 'EmpPost', 'Employment Postal Code [Source Data Type: varchar]' UNION ALL
      SELECT 'MailPost', 'Mail Postal Code [Source Data Type: varchar]' UNION ALL
      SELECT 'Occupation', 'Occupation [Source Data Type: varchar]' UNION ALL
      SELECT 'PayslipTyp', 'PayslipTyp [Source Data Type: varchar]' UNION ALL
      SELECT 'Academic', 'Academic qualifications [Source Data Type: varchar]' UNION ALL
      SELECT 'Emp_NM', 'Employer Name [Source Data Type: varchar]' UNION ALL
      SELECT 'SelftEmp_FG', 'Self Employment Indicator [Source Data Type: varchar]' UNION ALL
      SELECT 'Felda_FG', 'Felda is a government program to help rural Malaysians. [Source Data Type: varchar]' UNION ALL
      SELECT 'JoinIncome_FG', 'Joint Income Indicator [Source Data Type: varchar]' UNION ALL
      SELECT 'RecvPromo_FG', 'Received promotion indicator [Source Data Type: varchar]' UNION ALL
      SELECT 'N_Age', 'Age [Source Data Type: numeric]' UNION ALL
      SELECT 'N_YrStay', 'Number of years living in current residence [Source Data Type: numeric]' UNION ALL
      SELECT 'N_YrJob', 'Number of years in current job [Source Data Type: numeric]' UNION ALL
      SELECT 'B_NetIncome', 'Net Income [Source Data Type: numeric]' UNION ALL
      SELECT 'B_GrossIncome', 'Gross Income [Source Data Type: numeric]' UNION ALL
      SELECT 'B_AnnualIncome', 'Annual Income [Source Data Type: numeric]' UNION ALL
      SELECT 'EmpSts', 'Employment Status [Source Data Type: int]'
    )
    SELECT STRING_AGG(
      FORMAT('`%s` OPTIONS(description = "%s")', c.column_name, COALESCE(d.d, CONCAT('Customer CIF attribute: ', c.column_name))),
      ', ' ORDER BY c.ordinal_position
    )
    FROM `acsm_bronze.INFORMATION_SCHEMA.COLUMNS` c
    LEFT JOIN dict d ON LOWER(c.column_name) = LOWER(d.col)
    WHERE c.table_name = 'm3CIF'
  );

  IF m3_col_defs IS NOT NULL THEN
    EXECUTE IMMEDIATE FORMAT("""
      CREATE OR REPLACE VIEW `acsm_bronze.m3CIF` (%s)
      OPTIONS(description = 'Customer latest status daily refresh table (Governed via Mock Metadata.xlsx | Sheet: T7 - m3CIF | GCP Lakehouse Apache Iceberg)')
      AS SELECT * FROM `%s.acsm_gcp_lakehouse_catalog.acsm_gcp_bronze.m3CIF`
    """, m3_col_defs, @@project_id);
  END IF;
END;

-- 2. Ensure `acsm_bronze.dimProduct` view has all column descriptions from Mock Metadata.xlsx (Sheet: T8 - dimProduct)
BEGIN
  DECLARE dp_col_defs STRING;
  SET dp_col_defs = (
    WITH dict AS (
      SELECT 'Expiry_DT' AS col, 'Card expiry date [Source Data Type: varchar]' AS d UNION ALL
      SELECT 'FirstSpend_DT', 'First spend date [Source Data Type: varchar]' UNION ALL
      SELECT 'Block_Code', 'If card is blocked [Source Data Type: varchar]' UNION ALL
      SELECT 'Block_Date', 'If card is blocked [Source Data Type: numeric]' UNION ALL
      SELECT 'CIC_Status', 'CIC status [Source Data Type: varchar]' UNION ALL
      SELECT 'Card_Status', 'Card status [Source Data Type: varchar]' UNION ALL
      SELECT 'AKPK_Status', 'AKPK status [Source Data Type: varchar]' UNION ALL
      SELECT 'Card_First_Emboss_Date', 'Physical card issuance date [Source Data Type: numeric]' UNION ALL
      SELECT 'Card_Emboss_Date', 'Card emboss date [Source Data Type: numeric]' UNION ALL
      SELECT 'Card_First_Activated_Date', 'Card first activated date [Source Data Type: numeric]' UNION ALL
      SELECT 'Card_Activated_Date', 'Card current activated date [Source Data Type: numeric]' UNION ALL
      SELECT 'CP_CL', 'Credit Purchase Total Limit [Source Data Type: numeric]' UNION ALL
      SELECT 'CP_CL_Available', 'Credit Purchase Available Limit [Source Data Type: numeric]' UNION ALL
      SELECT 'CA_CL', 'Credit Advance Total Limit [Source Data Type: numeric]' UNION ALL
      SELECT 'CA_CL_Available', 'Credit Advance Available Limit [Source Data Type: numeric]' UNION ALL
      SELECT 'CP_CL_Usage', 'Credit Purchase Usage [Source Data Type: numeric]' UNION ALL
      SELECT 'CA_CL_Usage', 'Credit Advance Usage [Source Data Type: numeric]' UNION ALL
      SELECT 'CIF_ID', 'Unique customer ID [Source Data Type: varchar]' UNION ALL
      SELECT 'Account_No', 'Card account number [Source Data Type: numeric]' UNION ALL
      SELECT 'Account_Agree_Sts', 'Account agreement status [Source Data Type: varchar]' UNION ALL
      SELECT 'Virtual_Card_Flag', 'Virtual card indicator [Source Data Type: varchar]' UNION ALL
      SELECT 'Wallet_Tier', 'Loyalty account Tier [Source Data Type: varchar]' UNION ALL
      SELECT 'Brand_Card_Type', 'Credit Card Brand and Tier Type [Federated AWS Glue Attribute]' UNION ALL
      SELECT 'VIP_Flag', 'VIP Cardholder Indicator Flag [Federated AWS Glue Attribute]'
    )
    SELECT STRING_AGG(
      FORMAT('`%s` OPTIONS(description = "%s")', c.column_name, COALESCE(d.d, CONCAT('Credit card product attribute: ', c.column_name))),
      ', ' ORDER BY c.ordinal_position
    )
    FROM `acsm_bronze.INFORMATION_SCHEMA.COLUMNS` c
    LEFT JOIN dict d ON LOWER(c.column_name) = LOWER(d.col)
    WHERE c.table_name = 'dimProduct'
  );

  IF dp_col_defs IS NOT NULL THEN
    EXECUTE IMMEDIATE FORMAT("""
      CREATE OR REPLACE VIEW `acsm_bronze.dimProduct` (%s)
      OPTIONS(description = 'The current and daily full refresh master record of all active cards (Governed via Mock Metadata.xlsx | Sheet: T8 - dimProduct | AWS Glue Federated Iceberg)')
      AS SELECT * FROM `%s.acsm_aws_federated_catalog.acsm_aws_bronze.dimproduct`
    """, dp_col_defs, @@project_id);
  END IF;
END;

-- 3. Audit Table & Column Description Coverage Across All 8 `acsm_bronze` Tables & Views
WITH tbl_meta AS (
  SELECT
    t.table_name,
    t.table_type AS object_type,
    TRIM(o.option_value, '"') AS table_description
  FROM `acsm_bronze.INFORMATION_SCHEMA.TABLES` t
  LEFT JOIN `acsm_bronze.INFORMATION_SCHEMA.TABLE_OPTIONS` o
    ON t.table_name = o.table_name AND o.option_name = 'description'
  WHERE t.table_name IN (
    'Fact_EP_Judge', 'Fact_EP_Sales', 'Fact_EP_Collection',
    'Fact_CC_Judge', 'Fact_CC_Sales', 'Fact_CC_Collection',
    'm3CIF', 'dimProduct'
  )
),
col_meta AS (
  SELECT
    table_name,
    COUNT(*) AS total_columns,
    COUNTIF(description IS NOT NULL AND LENGTH(TRIM(description)) > 0) AS described_columns
  FROM `acsm_bronze.INFORMATION_SCHEMA.COLUMN_FIELD_PATHS`
  WHERE table_name IN (
    'Fact_EP_Judge', 'Fact_EP_Sales', 'Fact_EP_Collection',
    'Fact_CC_Judge', 'Fact_CC_Sales', 'Fact_CC_Collection',
    'm3CIF', 'dimProduct'
  )
  GROUP BY table_name
)
SELECT
  t.table_name,
  t.object_type,
  IF(t.table_description IS NOT NULL AND LENGTH(t.table_description) > 0, '✅ YES', '❌ NO') AS has_table_description,
  c.total_columns,
  c.described_columns,
  ROUND(c.described_columns * 100.0 / c.total_columns, 1) AS column_desc_coverage_pct,
  SUBSTR(t.table_description, 1, 90) AS table_description_preview
FROM tbl_meta t
INNER JOIN col_meta c USING (table_name)
ORDER BY t.object_type, t.table_name;

---
## Step 2: Pre-Flight Check — Verify the 8 Source Tables in `acsm_bronze` (`%%bigquery`)
Before creating the Silver and Gold Medallion layers, run the `%%bigquery` SQL cell below to verify that all **8 core ACSM tables** (`Fact_EP_Judge`, `Fact_EP_Sales`, `Fact_EP_Collection`, `Fact_CC_Judge`, `Fact_CC_Sales`, `Fact_CC_Collection`, `m3CIF`, `dimProduct`) are loaded in `acsm_bronze`.

In [ ]:
%%bigquery --project $PROJECT_ID --location $LOCATION
-- Verify all 8 source tables & views in acsm_bronze (6 Native Fact Tables + m3CIF Lakehouse Iceberg View + dimProduct AWS Glue Federated Iceberg View)
SELECT 'Fact_EP_Judge'      AS table_name, 'BigQuery Native Table'                AS storage_engine, COUNT(*) AS bronze_row_count FROM `acsm_bronze.Fact_EP_Judge`
UNION ALL
SELECT 'Fact_EP_Sales'      AS table_name, 'BigQuery Native Table'                AS storage_engine, COUNT(*) AS bronze_row_count FROM `acsm_bronze.Fact_EP_Sales`
UNION ALL
SELECT 'Fact_EP_Collection' AS table_name, 'BigQuery Native Table'                AS storage_engine, COUNT(*) AS bronze_row_count FROM `acsm_bronze.Fact_EP_Collection`
UNION ALL
SELECT 'Fact_CC_Judge'      AS table_name, 'BigQuery Native Table'                AS storage_engine, COUNT(*) AS bronze_row_count FROM `acsm_bronze.Fact_CC_Judge`
UNION ALL
SELECT 'Fact_CC_Sales'      AS table_name, 'BigQuery Native Table'                AS storage_engine, COUNT(*) AS bronze_row_count FROM `acsm_bronze.Fact_CC_Sales`
UNION ALL
SELECT 'Fact_CC_Collection' AS table_name, 'BigQuery Native Table'                AS storage_engine, COUNT(*) AS bronze_row_count FROM `acsm_bronze.Fact_CC_Collection`
UNION ALL
SELECT 'm3CIF'              AS table_name, 'GCP Lakehouse Iceberg View (GCS)'     AS storage_engine, COUNT(*) AS bronze_row_count FROM `acsm_bronze.m3CIF`
UNION ALL
SELECT 'dimProduct'         AS table_name, 'AWS Glue Federated Iceberg View (S3)' AS storage_engine, COUNT(*) AS bronze_row_count FROM `acsm_bronze.dimProduct`
ORDER BY table_name;

---
## Step 3: Create the Target Medallion Datasets (`acsm_silver` & `acsm_gold`) + Pre-Train BQML Model for Batch Pipeline Scoring
1. The **BigQuery Data Engineering Agent** requires the target datasets (`acsm_silver` and `acsm_gold`) to exist in the same region (`asia-southeast1` Singapore) as `acsm_bronze` before generating and running the pipeline.
2. Because BigQuery Pipeline Builder nodes execute `SELECT` queries (`CREATE OR REPLACE TABLE ... AS SELECT ...`), we also pre-train a fast in-database **BigQuery ML Credit Delinquency Propensity Model (`acsm_silver.model_delinquency_propensity`)** here so that our **Data Engineering Agent Pipeline** in **Step 4** can invoke **`ML.PREDICT` in batch mode** as the final Gold node!

Run the `%%bigquery` SQL cell below to create both datasets and train `acsm_silver.model_delinquency_propensity` (~8 seconds).

In [ ]:
%%bigquery --project $PROJECT_ID --location $LOCATION
-- 1. Create the Silver Medallion Dataset in Singapore (asia-southeast1)
CREATE SCHEMA IF NOT EXISTS `acsm_silver`
OPTIONS (
  location = 'asia-southeast1',
  description = 'ACSM Silver Medallion Layer: Standardized, type-cast, and deduplicated Customer CIF, EP/CC Underwriting, and Collections tables in Singapore (asia-southeast1)'
);

-- 2. Create the Gold Medallion Dataset in Singapore (asia-southeast1)
CREATE SCHEMA IF NOT EXISTS `acsm_gold`
OPTIONS (
  location = 'asia-southeast1',
  description = 'ACSM Gold Medallion Layer: Unified AEON 360 Customer Risk, Affordability & Credit Exposure Feature Store in Singapore (asia-southeast1)'
);

-- 3. Pre-train a fast BigQuery ML Logistic Regression Model (`acsm_silver.model_delinquency_propensity`)
--    so the Data Engineering Agent Pipeline in Step 4 can run Batch ML Scoring (`ML.PREDICT`) in `acsm_gold`!
CREATE OR REPLACE MODEL `acsm_silver.model_delinquency_propensity`
OPTIONS (
  MODEL_TYPE = 'LOGISTIC_REG',
  INPUT_LABEL_COLS = ['delinquency_risk_flag'],
  AUTO_CLASS_WEIGHTS = TRUE,
  MAX_ITERATIONS = 5
) AS
WITH col_flags AS (
  SELECT CAST(CIF_No AS STRING) AS CIF_ID, SUM(CAST(Unpaid_OSP AS NUMERIC)) AS unpaid_osp
  FROM (
    SELECT CIF_No, Unpaid_OSP FROM `acsm_bronze.Fact_EP_Collection`
    UNION ALL
    SELECT CIF_No, Unpaid_OSP FROM `acsm_bronze.Fact_CC_Collection`
  )
  GROUP BY 1
)
SELECT
  CAST(c.N_Age AS INT64) AS N_Age,
  CAST(c.B_NetIncome AS NUMERIC) AS B_NetIncome,
  CAST(c.B_AnnualIncome AS NUMERIC) AS B_AnnualIncome,
  TRIM(CAST(c.State AS STRING)) AS State,
  TRIM(CAST(c.Region AS STRING)) AS Region,
  TRIM(CAST(c.Occupation AS STRING)) AS Occupation,
  IF(COALESCE(f.unpaid_osp, 0) > 0, 1, 0) AS delinquency_risk_flag
FROM `acsm_bronze.m3CIF` c
LEFT JOIN col_flags f
  ON CAST(c.CIF_ID AS STRING) = f.CIF_ID;

-- 4. Verify all 3 Medallion Datasets (Bronze, Silver, Gold) in INFORMATION_SCHEMA
SELECT
  schema_name AS dataset_name,
  location,
  creation_time
FROM `region-asia-southeast1`.INFORMATION_SCHEMA.SCHEMATA
WHERE schema_name IN ('acsm_bronze', 'acsm_silver', 'acsm_gold')
ORDER BY schema_name;

---
## Step 4: How & Where to Copy-Paste the Medallion + Incremental + Assertions + Batch BQML Prompt in the BigQuery Data Engineering Agent

Now that `acsm_bronze` (with **100% table & column descriptions across all 8 tables and views** verified in **Step 0**), `acsm_silver` (with the pre-trained BQML model `acsm_silver.model_delinquency_propensity`), and `acsm_gold` are ready in **`asia-southeast1` (Singapore)**, follow these clicks in **BigQuery Studio** to generate your **Medallion + Incremental Load + Dataform Assertions + Batch BQML Scoring Pipeline DAG** using natural language.

### 🧭 Part A: Where to Click in BigQuery Studio
1. Open **Google Cloud Console** $\rightarrow$ navigate to **BigQuery** $\rightarrow$ **Studio**.
2. In the left **Explorer** pane, expand your project (`${PROJECT_ID}`) and verify you see the three datasets:
   - `acsm_bronze` *(contains all 8 source tables & views enriched with 100% `Mock Metadata.xlsx` table & column descriptions)*
   - `acsm_silver` *(created in Step 3 + contains BQML model `model_delinquency_propensity`)*
   - `acsm_gold` *(created in Step 3)*
3. At the top of the **BigQuery Studio workspace tab bar** (next to **`+ SQL query`** and **`+ Notebook`**), click **`+` (Create new)** $\rightarrow$ select **`Pipeline`**.
4. When prompted for the **Pipeline Location / Code Region**, select **`asia-southeast1 (Singapore)`**.
   - *Tip*: If BigQuery displays a banner asking to grant the **Dataform Service Agent** (`service-<PROJECT_NUMBER>@gcp-sa-dataform.iam.gserviceaccount.com`) access to BigQuery, click **`Grant`**.
5. Inside the **Pipeline Canvas**, click the **Gemini Sparkle button (`Ask Data Engineering Agent` / `Generate with Gemini`)**.

---
### 💬 Part B (Option 1 — Recommended for Live Demo): High-Level Business + Incremental + Dataform Assertions + Batch BQML Prompt
Because **Step 0** confirmed that all 8 `acsm_bronze` tables and views carry **100% Dataplex table & column descriptions**, the **Data Engineering Agent** can generate the Medallion transformations, **incremental timestamp filtering**, **Dataform primary-key / non-null assertions**, and **Batch BQML predictions (`ML.PREDICT`)** from this single natural-language prompt:

```text
Using the source tables and views in the `acsm_bronze` dataset, build a 3-tier Medallion and Batch ML scoring pipeline into `acsm_silver` and `acsm_gold`:

1. In `acsm_silver`, create four cleansed domain tables:
   - `silver_customer_cif`: Deduplicate customer master records from `acsm_bronze.m3CIF` by customer ID (`CIF_ID`), keeping the most recent record based on `Rcd_DT`, with standardized demographic and income fields (`N_Age`, `B_NetIncome`, `B_AnnualIncome`, `State`, `Region`, `Occupation`).
   - `silver_ep_underwriting`: Cleanse Easy Payment loan applications from `acsm_bronze.Fact_EP_Judge`, standardize the customer ID and application date (`APPL_DT`), and retain key underwriting score, DSR, NDI, and financing amount metrics.
   - `silver_cc_underwriting`: Cleanse Credit Card applications from `acsm_bronze.Fact_CC_Judge`, standardize the application date (`Appl_DT`), and retain credit score, DSR, NDI, and approved credit limit metrics.
   - `silver_collections_summary`: Combine Easy Payment (`Fact_EP_Collection`) and Credit Card (`Fact_CC_Collection`) collections at the customer level to calculate total unpaid principal and worst collection score grade per customer.

2. Build incremental load logic with timestamp/date filtering (`Rcd_DT`, `APPL_DT`, `Appl_DT`, `TX_DT`) for daily underwriting and collection updates to reduce compute footprint during recurring pipeline executions.

3. Add Dataform assertion tests for unique primary keys (`CIF_ID`, `APPL_NO`, `Appl_ID`) and non-null constraints across incremental silver tables to safeguard data integrity.

4. In `acsm_gold`, create an executive Customer 360 table `acsm_gold.gold_aeon360_customer_profile` by joining `silver_customer_cif` with customer-level summaries from `silver_ep_underwriting`, `silver_cc_underwriting`, `silver_collections_summary`, and active credit card usage from `acsm_bronze.dimProduct`.

5. In `acsm_gold`, create a batch ML prediction table `acsm_gold.gold_aeon360_batch_ml_predictions` by running BigQuery ML batch inference (`ML.PREDICT`) with model `acsm_silver.model_delinquency_propensity` over `acsm_gold.gold_aeon360_customer_profile` to score all customers with predicted delinquency risk.
```

---
### 🔧 Part B (Option 2 — Schema-Pinned Prompt): Exact Column Aliases + Incremental + Assertions + Batch `ML.PREDICT`
If you want the Data Engineering Agent to output the **exact column aliases** used by the **Step 6 Reconciliation Audit** and **Notebook 03 (Governance & Security)** while also including incremental watermarks, Dataform assertion tests, and the Batch BQML `ML.PREDICT` node, use this prompt:

```text
Build a Medallion architecture data transformation and batch ML scoring pipeline in BigQuery (location asia-southeast1) using source tables and views from the `acsm_bronze` dataset to populate standardized tables in `acsm_silver` and Customer 360 + Batch BQML prediction tables in `acsm_gold`:

1. Create Silver table `acsm_silver.silver_customer_cif` from `acsm_bronze.m3CIF`:
   - Deduplicate records by `CIF_ID` keeping the latest record based on `Rcd_DT`.
   - Select `CIF_ID`, `CIF_NM`, `Gender`, `MaritalSts`, `Citizen`, `State`, `Region`, `Race`, `Occupation`, `EmpSts`, `N_Age`, `N_YrStay`, `N_YrJob`,
     CAST(`B_NetIncome` AS NUMERIC) AS `B_NetIncome`,
     CAST(`B_GrossIncome` AS NUMERIC) AS `B_GrossIncome`,
     CAST(`B_AnnualIncome` AS NUMERIC) AS `B_AnnualIncome`,
     and `RecvPromo_FG`.

2. Create Silver table `acsm_silver.silver_ep_underwriting` from `acsm_bronze.Fact_EP_Judge`:
   - Standardize `CIF_NO` as `CIF_ID`.
   - Parse integer date `APPL_DT` (format YYYYMMDD) into a BigQuery DATE column `application_date` using SAFE.PARSE_DATE('%Y%m%d', CAST(APPL_DT AS STRING)).
   - Include `APPL_NO`, `AGREE_NO`, `CIF_ID`, `application_date`, `APPL_STS`, `SCORING_POINT`, `SCORING_RANK`, `SCORE_DECISION`, `LOAN_GRP`, `FIN_AMT`, `INST_AMT`, `INTEREST`, `TOTAL_INST`, `NetIncome`, `NDI`, `CUR_DSR`, `NEW_DSR`, and `TOTAL_AEON_OSB`.

3. Create Silver table `acsm_silver.silver_cc_underwriting` from `acsm_bronze.Fact_CC_Judge`:
   - Parse integer date `Appl_DT` (format YYYYMMDD) into a BigQuery DATE column `application_date` using SAFE.PARSE_DATE('%Y%m%d', CAST(Appl_DT AS STRING)).
   - Include `Appl_ID`, `Account_No`, `CIF_ID`, `application_date`, `ApplSts_ID`, `CardTyp_ID`, `CardBrand_ID`, `ScoreDecision_ID`, `ScoreRank_ID`, `NetIncome`, `NDI`, `CurrDSR`, `NewDSR`, `B_CrLimit`, `Final_Score`, and `Final_ScoreDesc`.

4. Create Silver table `acsm_silver.silver_collections_summary` by combining `acsm_bronze.Fact_EP_Collection` and `acsm_bronze.Fact_CC_Collection`:
   - Aggregate at the customer level (`CIF_No` aliased as `CIF_ID`) to compute:
     `total_ep_unpaid_osp` (SUM of Unpaid_OSP from `Fact_EP_Collection`),
     `total_cc_unpaid_osp` (SUM of Unpaid_OSP from `Fact_CC_Collection`),
     `combined_unpaid_osp` (total_ep_unpaid_osp + total_cc_unpaid_osp),
     `worst_collection_score_grade` (MAX of Score_Grade across both tables).

5. Build incremental load logic with timestamp filtering for daily underwriting and collection updates to reduce compute footprint during recurring pipeline executions.

6. Add Dataform assertion tests for unique primary keys and non-null constraints across incremental silver tables to safeguard data integrity.

7. Create Gold table `acsm_gold.gold_aeon360_customer_profile`:
   - Join `acsm_silver.silver_customer_cif` (base customer table) with:
     a) Aggregated `acsm_silver.silver_ep_underwriting` per `CIF_ID` (total EP applications `ep_app_count`, total financed amount `total_ep_financed_myr`, average EP new DSR `avg_ep_new_dsr`),
     b) Aggregated `acsm_silver.silver_cc_underwriting` per `CIF_ID` (total CC applications `cc_app_count`, total approved credit limit `total_cc_limit_myr`, latest CTOS score `latest_ctos_score`),
     c) `acsm_silver.silver_collections_summary` per `CIF_ID` (`combined_unpaid_osp`, `worst_collection_score_grade`),
     d) Aggregated `acsm_bronze.dimProduct` per `CIF_ID` (count of active credit cards `active_card_count`, total credit purchase usage `total_cp_usage_myr`, total credit purchase available `total_cp_available_myr`).
   - Cluster `acsm_gold.gold_aeon360_customer_profile` by `State` and `CIF_ID`.

8. Create Gold batch ML scoring table `acsm_gold.gold_aeon360_batch_ml_predictions`:
   - Run `SELECT * FROM ML.PREDICT(MODEL \`acsm_silver.model_delinquency_propensity\`, TABLE \`acsm_gold.gold_aeon360_customer_profile\`)` to batch-score all 100,000 customers with `predicted_delinquency_risk_flag` and class probabilities as the final downstream node of the pipeline.
```

---
### ▶️ Part C: Review the Visual DAG & Run the Pipeline
1. The **Data Engineering Agent** will generate your visual **Dataform Medallion DAG** on the Pipeline Canvas:
   - **4 Silver transformation nodes** (`silver_customer_cif`, `silver_ep_underwriting`, `silver_cc_underwriting`, `silver_collections_summary`) with incremental logic & Dataform assertion checks (`uniqueKey`, `nonNull`).
   - **1 Gold Customer 360 join node** (`gold_aeon360_customer_profile`) waiting for all 4 Silver tables + `dimProduct`.
   - **1 Gold Batch BQML Scoring node** (`gold_aeon360_batch_ml_predictions`) running `ML.PREDICT` over `gold_aeon360_customer_profile`!
2. Click any node on the canvas to inspect the generated SQL/SQLX definition and assertion rules.
3. Click **`Apply`** to accept the agent's generated nodes, then click **`Run`** at the top of the Pipeline Canvas (or run **Step 5** below to materialize the exact production schema for **Step 6** and **Notebook 03**).

---
## Step 5: Transparent `%%bigquery` SQL Medallion Execution (Direct SQL Option & Reconciliation)
In addition to (or as a direct SQL comparison to) running the Pipeline Canvas in Step 4, you can execute the **exact same 5 Silver & Gold Medallion table transformations + Dual-Run Financial Reconciliation Audit** right here in this notebook using `%%bigquery` SQL so every line of transformation logic is 100% transparent.

In [ ]:
%%bigquery --project $PROJECT_ID --location $LOCATION
-- =============================================================================
-- 1. SILVER LAYER: acsm_silver.silver_customer_cif (Deduplicated Customer Master)
-- =============================================================================
CREATE OR REPLACE TABLE `acsm_silver.silver_customer_cif`
CLUSTER BY CIF_ID, State
OPTIONS (
  description = 'Governed Silver Customer Master (m3CIF) deduplicated by CIF_ID with typed income and PDPA consent flags.'
) AS
SELECT
  CAST(CIF_ID AS STRING) AS CIF_ID,
  SAFE.PARSE_DATE('%Y-%m-%d', SUBSTR(CAST(Rcd_DT AS STRING), 1, 10)) AS record_refresh_date,
  TRIM(CAST(CIF_NM AS STRING)) AS CIF_NM,
  TRIM(CAST(Gender AS STRING)) AS Gender,
  TRIM(CAST(MaritalSts AS STRING)) AS MaritalSts,
  TRIM(CAST(Citizen AS STRING)) AS Citizen,
  TRIM(CAST(State AS STRING)) AS State,
  TRIM(CAST(Region AS STRING)) AS Region,
  TRIM(CAST(Race AS STRING)) AS Race,
  TRIM(CAST(Occupation AS STRING)) AS Occupation,
  CAST(EmpSts AS INT64) AS EmpSts,
  CAST(N_Age AS INT64) AS N_Age,
  CAST(N_YrStay AS NUMERIC) AS N_YrStay,
  CAST(N_YrJob AS NUMERIC) AS N_YrJob,
  CAST(B_NetIncome AS NUMERIC) AS B_NetIncome,
  CAST(B_GrossIncome AS NUMERIC) AS B_GrossIncome,
  CAST(B_AnnualIncome AS NUMERIC) AS B_AnnualIncome,
  COALESCE(TRIM(CAST(RecvPromo_FG AS STRING)), 'N') AS RecvPromo_FG
FROM `acsm_bronze.m3CIF`
QUALIFY ROW_NUMBER() OVER (PARTITION BY CAST(CIF_ID AS STRING) ORDER BY Rcd_DT DESC) = 1;

-- =============================================================================
-- 2. SILVER LAYER: acsm_silver.silver_ep_underwriting (Easy Payment Underwriting)
-- =============================================================================
CREATE OR REPLACE TABLE `acsm_silver.silver_ep_underwriting`
CLUSTER BY CIF_ID, APPL_STS
OPTIONS (
  description = 'Governed Silver Easy Payment (EP) Application & Underwriting Decisions from acsm_bronze.Fact_EP_Judge.'
) AS
SELECT
  CAST(APPL_NO AS STRING) AS APPL_NO,
  CAST(AGREE_NO AS STRING) AS AGREE_NO,
  CAST(CIF_NO AS STRING) AS CIF_ID,
  SAFE.PARSE_DATE('%Y%m%d', NULLIF(TRIM(CAST(APPL_DT AS STRING)), '0')) AS application_date,
  TRIM(CAST(APPL_STS AS STRING)) AS APPL_STS,
  CAST(SCORING_POINT AS NUMERIC) AS SCORING_POINT,
  TRIM(CAST(SCORING_RANK AS STRING)) AS SCORING_RANK,
  TRIM(CAST(SCORE_DECISION AS STRING)) AS SCORE_DECISION,
  TRIM(CAST(LOAN_GRP AS STRING)) AS LOAN_GRP,
  CAST(FIN_AMT AS NUMERIC) AS FIN_AMT,
  CAST(INST_AMT AS NUMERIC) AS INST_AMT,
  CAST(INTEREST AS NUMERIC) AS INTEREST,
  CAST(TOTAL_INST AS INT64) AS TOTAL_INST,
  CAST(NetIncome AS NUMERIC) AS NetIncome,
  CAST(NDI AS NUMERIC) AS NDI,
  CAST(CUR_DSR AS NUMERIC) AS CUR_DSR,
  CAST(NEW_DSR AS NUMERIC) AS NEW_DSR,
  CAST(TOTAL_AEON_OSB AS NUMERIC) AS TOTAL_AEON_OSB
FROM `acsm_bronze.Fact_EP_Judge`;

-- =============================================================================
-- 3. SILVER LAYER: acsm_silver.silver_cc_underwriting (Credit Card Underwriting)
-- =============================================================================
CREATE OR REPLACE TABLE `acsm_silver.silver_cc_underwriting`
CLUSTER BY CIF_ID, ApplSts_ID
OPTIONS (
  description = 'Governed Silver Credit Card Application & Underwriting Decisions from acsm_bronze.Fact_CC_Judge.'
) AS
SELECT
  CAST(Appl_ID AS STRING) AS Appl_ID,
  CAST(Account_No AS STRING) AS Account_No,
  CAST(CIF_ID AS STRING) AS CIF_ID,
  SAFE.PARSE_DATE('%Y%m%d', NULLIF(TRIM(CAST(Appl_DT AS STRING)), '0')) AS application_date,
  TRIM(CAST(ApplSts_ID AS STRING)) AS ApplSts_ID,
  TRIM(CAST(CardTyp_ID AS STRING)) AS CardTyp_ID,
  TRIM(CAST(CardBrand_ID AS STRING)) AS CardBrand_ID,
  TRIM(CAST(ScoreDecision_ID AS STRING)) AS ScoreDecision_ID,
  TRIM(CAST(ScoreRank_ID AS STRING)) AS ScoreRank_ID,
  CAST(NetIncome AS NUMERIC) AS NetIncome,
  CAST(NDI AS NUMERIC) AS NDI,
  CAST(CurrDSR AS NUMERIC) AS CurrDSR,
  CAST(NewDSR AS NUMERIC) AS NewDSR,
  CAST(B_CrLimit AS NUMERIC) AS B_CrLimit,
  CAST(Final_Score AS NUMERIC) AS Final_Score,
  TRIM(CAST(Final_ScoreDesc AS STRING)) AS Final_ScoreDesc
FROM `acsm_bronze.Fact_CC_Judge`;

-- =============================================================================
-- 4. SILVER LAYER: acsm_silver.silver_collections_summary (EP + CC Collections)
-- =============================================================================
CREATE OR REPLACE TABLE `acsm_silver.silver_collections_summary`
CLUSTER BY CIF_ID
OPTIONS (
  description = 'Customer-level Silver Collections & Delinquency Summary across Fact_EP_Collection and Fact_CC_Collection.'
) AS
WITH ep_col AS (
  SELECT
    CAST(CIF_No AS STRING) AS CIF_ID,
    SUM(CAST(Unpaid_OSP AS NUMERIC)) AS total_ep_unpaid_osp,
    MAX(TRIM(CAST(Score_Grade AS STRING))) AS ep_worst_grade
  FROM `acsm_bronze.Fact_EP_Collection`
  GROUP BY 1
),
cc_col AS (
  SELECT
    CAST(CIF_No AS STRING) AS CIF_ID,
    SUM(CAST(Unpaid_OSP AS NUMERIC)) AS total_cc_unpaid_osp,
    MAX(TRIM(CAST(Score_Grade AS STRING))) AS cc_worst_grade
  FROM `acsm_bronze.Fact_CC_Collection`
  GROUP BY 1
)
SELECT
  COALESCE(ep.CIF_ID, cc.CIF_ID) AS CIF_ID,
  COALESCE(ep.total_ep_unpaid_osp, 0) AS total_ep_unpaid_osp,
  COALESCE(cc.total_cc_unpaid_osp, 0) AS total_cc_unpaid_osp,
  COALESCE(ep.total_ep_unpaid_osp, 0) + COALESCE(cc.total_cc_unpaid_osp, 0) AS combined_unpaid_osp,
  GREATEST(COALESCE(ep.ep_worst_grade, 'A'), COALESCE(cc.cc_worst_grade, 'A')) AS worst_collection_score_grade
FROM ep_col ep
FULL OUTER JOIN cc_col cc
  ON ep.CIF_ID = cc.CIF_ID;

-- =============================================================================
-- 5. GOLD LAYER: acsm_gold.gold_aeon360_customer_profile (AEON 360 Feature Store)
-- =============================================================================
CREATE OR REPLACE TABLE `acsm_gold.gold_aeon360_customer_profile`
CLUSTER BY State, CIF_ID
OPTIONS (
  description = 'Gold AEON 360 Customer Risk, Affordability & Credit Exposure Feature Store joining CIF, EP Underwriting, CC Underwriting, Collections, and Card Utilization.'
) AS
WITH ep_agg AS (
  SELECT
    CIF_ID,
    COUNT(*) AS ep_app_count,
    ROUND(SUM(COALESCE(FIN_AMT, 0)), 2) AS total_ep_financed_myr,
    ROUND(AVG(NEW_DSR), 2) AS avg_ep_new_dsr
  FROM `acsm_silver.silver_ep_underwriting`
  GROUP BY CIF_ID
),
cc_agg AS (
  SELECT
    CIF_ID,
    COUNT(*) AS cc_app_count,
    ROUND(SUM(COALESCE(B_CrLimit, 0)), 2) AS total_cc_limit_myr,
    MAX(Final_Score) AS latest_ctos_score
  FROM `acsm_silver.silver_cc_underwriting`
  GROUP BY CIF_ID
),
card_agg AS (
  SELECT
    CAST(CIF_ID AS STRING) AS CIF_ID,
    COUNTIF(TRIM(CAST(Card_Status AS STRING)) = 'Active') AS active_card_count,
    ROUND(SUM(CAST(CP_CL_Usage AS NUMERIC)), 2) AS total_cp_usage_myr,
    ROUND(SUM(CAST(CP_CL_Available AS NUMERIC)), 2) AS total_cp_available_myr
  FROM `acsm_bronze.dimProduct`
  GROUP BY 1
)
SELECT
  c.CIF_ID,
  c.CIF_NM,
  c.State,
  c.Region,
  c.Occupation,
  c.N_Age,
  c.B_NetIncome,
  c.B_AnnualIncome,
  c.RecvPromo_FG,
  COALESCE(ep.ep_app_count, 0) AS ep_app_count,
  COALESCE(ep.total_ep_financed_myr, 0) AS total_ep_financed_myr,
  ep.avg_ep_new_dsr,
  COALESCE(cc.cc_app_count, 0) AS cc_app_count,
  COALESCE(cc.total_cc_limit_myr, 0) AS total_cc_limit_myr,
  cc.latest_ctos_score,
  COALESCE(col.total_ep_unpaid_osp, 0) AS total_ep_unpaid_osp,
  COALESCE(col.total_cc_unpaid_osp, 0) AS total_cc_unpaid_osp,
  COALESCE(col.combined_unpaid_osp, 0) AS combined_unpaid_osp,
  COALESCE(col.worst_collection_score_grade, 'NONE') AS worst_collection_score_grade,
  COALESCE(crd.active_card_count, 0) AS active_card_count,
  COALESCE(crd.total_cp_usage_myr, 0) AS total_cp_usage_myr,
  COALESCE(crd.total_cp_available_myr, 0) AS total_cp_available_myr
FROM `acsm_silver.silver_customer_cif` c
LEFT JOIN ep_agg ep USING (CIF_ID)
LEFT JOIN cc_agg cc USING (CIF_ID)
LEFT JOIN `acsm_silver.silver_collections_summary` col USING (CIF_ID)
LEFT JOIN card_agg crd USING (CIF_ID);

-- =============================================================================
-- 6. GOLD LAYER (BATCH BQML INFERENCE NODE): acsm_gold.gold_aeon360_batch_ml_predictions
-- =============================================================================
CREATE OR REPLACE TABLE `acsm_gold.gold_aeon360_batch_ml_predictions`
CLUSTER BY State, CIF_ID
OPTIONS (
  description = 'Gold Batch BQML Delinquency Risk Predictions scoring all 100,000 customers via ML.PREDICT(MODEL acsm_silver.model_delinquency_propensity).'
) AS
SELECT
  *
FROM ML.PREDICT(
  MODEL `acsm_silver.model_delinquency_propensity`,
  TABLE `acsm_gold.gold_aeon360_customer_profile`
);

---
## Step 6: Verify `acsm_silver` & `acsm_gold` Tables and Dual-Run Financial Reconciliation (`%%bigquery`)
Run the two `%%bigquery` SQL cells below to:
1. Inspect the created tables and row counts in `acsm_silver` and `acsm_gold`.
2. Run the **Dual-Run Financial Control Total Reconciliation Audit** (`Bronze vs. Silver/Gold`) to prove **0.00 MYR variance** across Easy Payment financed principal (`FIN_AMT`) and Collections unpaid principal (`Unpaid_OSP`).

In [ ]:
%%bigquery --project $PROJECT_ID --location $LOCATION
-- 1. Verify all created Medallion tables across acsm_silver and acsm_gold
SELECT
  table_schema AS medallion_layer,
  table_name,
  SUM(total_rows) AS total_rows,
  ROUND(SUM(total_logical_bytes) / (1024 * 1024), 2) AS logical_mb
FROM `acsm_silver.INFORMATION_SCHEMA.PARTITIONS`
GROUP BY table_schema, table_name
UNION ALL
SELECT
  table_schema AS medallion_layer,
  table_name,
  SUM(total_rows) AS total_rows,
  ROUND(SUM(total_logical_bytes) / (1024 * 1024), 2) AS logical_mb
FROM `acsm_gold.INFORMATION_SCHEMA.PARTITIONS`
GROUP BY table_schema, table_name
ORDER BY medallion_layer, table_name;

In [ ]:
%%bigquery --project $PROJECT_ID --location $LOCATION
-- 2. Dual-Run Financial Control Total Reconciliation Audit (Bronze vs. Silver) + Preview Gold AEON 360
WITH checks AS (
  SELECT
    'Fact_EP_Judge -> silver_ep_underwriting' AS pipeline_flow,
    'FIN_AMT (Financed Principal MYR)' AS control_metric,
    (SELECT COUNT(*) FROM `acsm_bronze.Fact_EP_Judge`) AS bronze_rows,
    (SELECT COUNT(*) FROM `acsm_silver.silver_ep_underwriting`) AS silver_rows,
    (SELECT ROUND(SUM(CAST(FIN_AMT AS NUMERIC)), 2) FROM `acsm_bronze.Fact_EP_Judge`) AS bronze_total_myr,
    (SELECT ROUND(SUM(FIN_AMT), 2) FROM `acsm_silver.silver_ep_underwriting`) AS silver_total_myr
  UNION ALL
  SELECT
    'Fact_CC_Judge -> silver_cc_underwriting' AS pipeline_flow,
    'B_CrLimit (Approved Credit Limit MYR)' AS control_metric,
    (SELECT COUNT(*) FROM `acsm_bronze.Fact_CC_Judge`) AS bronze_rows,
    (SELECT COUNT(*) FROM `acsm_silver.silver_cc_underwriting`) AS silver_rows,
    (SELECT ROUND(SUM(CAST(B_CrLimit AS NUMERIC)), 2) FROM `acsm_bronze.Fact_CC_Judge`) AS bronze_total_myr,
    (SELECT ROUND(SUM(B_CrLimit), 2) FROM `acsm_silver.silver_cc_underwriting`) AS silver_total_myr
  UNION ALL
  SELECT
    'Fact_EP_Collection + Fact_CC_Collection -> silver_collections_summary' AS pipeline_flow,
    'Unpaid_OSP (Combined Unpaid Principal MYR)' AS control_metric,
    (SELECT COUNT(DISTINCT CIF_No) FROM (
      SELECT CIF_No FROM `acsm_bronze.Fact_EP_Collection`
      UNION DISTINCT
      SELECT CIF_No FROM `acsm_bronze.Fact_CC_Collection`
    )) AS bronze_rows,
    (SELECT COUNT(*) FROM `acsm_silver.silver_collections_summary`) AS silver_rows,
    (SELECT ROUND(SUM(CAST(Unpaid_OSP AS NUMERIC)), 2) FROM `acsm_bronze.Fact_EP_Collection`) +
      (SELECT ROUND(SUM(CAST(Unpaid_OSP AS NUMERIC)), 2) FROM `acsm_bronze.Fact_CC_Collection`) AS bronze_total_myr,
    (SELECT ROUND(SUM(combined_unpaid_osp), 2) FROM `acsm_silver.silver_collections_summary`) AS silver_total_myr
)
SELECT
  pipeline_flow,
  control_metric,
  bronze_rows,
  silver_rows,
  bronze_total_myr,
  silver_total_myr,
  (silver_total_myr - bronze_total_myr) AS variance_myr,
  IF(bronze_rows = silver_rows AND ABS(silver_total_myr - bronze_total_myr) = 0, 'PASS (0.00 MYR VARIANCE)', 'INVESTIGATE') AS audit_status
FROM checks;

In [ ]:
%%bigquery --project $PROJECT_ID --location $LOCATION
-- 3. Preview Top 10 Customers in the Gold Batch BQML Predictions Table (`acsm_gold.gold_aeon360_batch_ml_predictions`)
SELECT
  CIF_ID,
  CIF_NM,
  State,
  B_NetIncome,
  total_ep_financed_myr,
  total_cc_limit_myr,
  combined_unpaid_osp,
  worst_collection_score_grade,
  predicted_delinquency_risk_flag,
  ROUND(predicted_delinquency_risk_flag_probs[OFFSET(0)].prob, 4) AS predicted_delinquency_prob
FROM `acsm_gold.gold_aeon360_batch_ml_predictions`
ORDER BY total_ep_financed_myr + total_cc_limit_myr DESC
LIMIT 10;